## `ip route` — The Routing Table & Default Gateway

**Support context:** `ip addr` tells you a machine has an IP. It says nothing about whether that machine knows _how to leave the local network_. "I can reach other machines on my LAN but nothing on the internet" is a routing symptom, not an addressing one — this is where you check.

**COMMAND:** `ip route`

**PURPOSE (support):** Every packet destined outside the local subnet needs a **next hop** — the default gateway. If that entry is missing, wrong, or points to a dead device, the box is isolated from anything off-subnet even though its own IP/interface look perfectly healthy.
> This is the second checkpoint after `ip addr`, before you even think about DNS or firewalls.

**SYNTAX:** `ip route [show]` (often abbreviated `ip r`)

**FLAGS:**

|Flag/Form|Meaning|
|---|---|
|`ip route` / `ip r`|show the full routing table|
|`ip route get <IP>`|show which route/interface would be used to reach a specific address — great for "would this even leave the box" checks|
|`ip -4 route`|IPv4 routes only|
|`ip -6 route`|IPv6 routes only|

**Developer bridge:** think of the routing table like Django's `URLconf` resolution order — the kernel checks routes top to bottom (most specific match wins) to decide where a packet goes, the same way Django walks `urlpatterns` looking for the first pattern that matches a request path. The **default route** (`default via ...`) is your `urlpatterns`' catch-all — "if nothing more specific matched, send it here."

**EXAMPLE:**

In [ ]:
dennie@X390:~/linux-tut$ ip route
default via 172.20.144.1 dev eth0 proto kernel
172.20.144.0/20 dev eth0 proto kernel scope link src 172.20.144.12

**READ THE OUTPUT:**

Two lines, two different jobs:

- **`default via 172.20.144.1 dev eth0 proto kernel`** — the **default route**. Anything not matched by a more specific line below gets sent to `172.20.144.1` (the gateway) out through `eth0`. **No line like this = no way off the local subnet, full stop**.
- **`172.20.144.0/20 dev eth0 proto kernel scope link src 172.20.144.12`** — the **local subnet route**, auto-created the moment `eth0` got its IP. `scope link` means "directly reachable, no gateway needed" — anything inside `172.20.144.0/20` goes straight out `eth0`, no hop required. `src 172.20.144.12` is which of your own addresses the kernel uses as the source when it sends from this route.

**`proto kernel`** on both lines means the kernel auto-generated these from interface config — you didn't add them by hand with `ip route add`. If you ever see `proto static` or `proto dhcp` instead, that tells you *how* the route got there, which matters when a route looks wrong and you're trying to figure out who/what is responsible for it.

> **Route selection isn't "top of the list wins."** The kernel uses **longest prefix match** — the most specific matching route wins regardless of print order. A `/32` route to one exact host will always beat the `/20` subnet route or the `default` (`/0`) route for that host, even if `ip route` happens to print it last. The `URLconf`-style "most specific match wins" framing from the intro is right — just don't read literal list order as priority order.

**TRY NOW:**

Run these three in your WSL terminal and paste the real output into cells below:

```
ip route
ip -br route
ip route get 8.8.8.8
```

`ip route get 8.8.8.8` won't send any traffic — it just asks the kernel "which route would you pick to reach this address," which is the fastest way to confirm routing behavior for one destination without generating real packets.

In [ ]:
dennie@X390:~/linux-tut$ ip route
default via 172.30.240.1 dev eth0 proto kernel
172.30.240.0/20 dev eth0 proto kernel scope link src 172.30.240.133

In [ ]:
dennie@X390:~/linux-tut$ ip -br route
default via 172.30.240.1 dev eth0 proto kernel
172.30.240.0/20 dev eth0 proto kernel scope link src 172.30.240.133

In [ ]:
dennie@X390:~/linux-tut$ ip route get 8.8.8.8
8.8.8.8 via 172.30.240.1 dev eth0 src 172.30.240.133 uid 1000
    cache

All three came back healthy and consistent with each other:

- **`ip route`** — `default via 172.30.240.1 dev eth0` plus the `172.30.240.0/20` link route. Clean routing table, nothing missing.
- **`ip -br route`** — printed identically to the plain version this time, and that's expected, not a bug. `-br` earns its keep when there's a lot to compress: several interfaces, VPN routes, static routes stacked up. With only two entries (default + one local subnet), there's nothing left to flatten — you'll see the real difference once your table has more than a couple of lines.
- **`ip route get 8.8.8.8`** — confirms the kernel would use `via 172.30.240.1 dev eth0 src 172.30.240.133` to reach `8.8.8.8`, i.e. out the default route as expected. `cache` at the bottom just means the kernel cached this lookup result — not something you need to act on.

> Notice your gateway here is `172.30.240.1`, not the `172.20.144.1` from the lesson's example — exactly the WSL NAT drift called out in the GOTCHA above. Live proof: never trust a gateway IP you remembered from a previous session, always re-run `ip route`.

**TICKET SCENARIO:** "User can reach other machines/printers on the office network fine, but says the internet itself is down — can't load any external site." You've already ruled out addressing (`ip addr` looked healthy — valid IP, `UP`, `LOWER_UP`). Next stop: `ip route`.
> If the `default via ...` line is missing entirely, or points to an IP that isn't actually the real gateway, that's your **root cause** — the box can talk to its own subnet (LAN traffic, matched by the `scope link` route) but has no path anywhere else.

**DIAGNOSIS PATH:**

1. `ip addr` — confirm the interface is `UP`/`LOWER_UP` with a sane IP (already step zero from the last lesson).
2. `ip route` — is there a `default via ...` line at all? Missing default route = total isolation from anything off-subnet, even though local traffic works fine.
3. Sanity-check the gateway IP itself — does `172.20.144.1` actually fall inside your subnet (`172.20.144.0/20`)? A gateway address outside your own subnet is invalid and will never work, no matter what the config file says.
4. `ping -c 3 172.20.144.1` — can you even reach the gateway itself? If the route looks correct but the gateway doesn't respond, the problem has moved from "routing config" to "the gateway device itself" or something blocking traffic to it (firewall, dead port, cable).
5. `ip route get <destination>` — for one specific complaint ("I can't reach 10.0.5.20"), this tells you exactly which route/interface the kernel would use, without needing to actually send traffic to test it.

**GOTCHA (WSL-specific):** Just like the `eth0` IP itself, WSL2's default gateway address is **not stable** — it's reassigned by the virtual NAT and commonly changes across `wsl --shutdown`/reboots or Windows networking changes. Never hardcode a gateway IP in a note or runbook and trust it later; always re-check live with `ip route` rather than relying on a value from a previous session.